# NyayaAI - Training Notebook

End-to-end pipeline for **IndiaLex-FT** — a LoRA fine-tuned LLM for Indian legal and income-tax Q&A.

**Steps covered:**
1. Install dependencies
2. Load API keys from `.env`
3. Baseline evaluation (pre-training)
4. LoRA SFT fine-tuning
5. Post-training evaluation
6. LLM-as-Judge scoring via claude-sonnet-4-6
7. Download outputs

> **Note:** Run this notebook from the `indialex-ft/` directory. Ensure your `.env` file has `ANTHROPIC_API_KEY` and `GROQ_API_KEY` set before starting.

In [1]:
import os
from pathlib import Path

# Ensure the notebook working directory is indialex-ft/
notebook_dir = Path("__file__" if "__file__" in dir() else ".").resolve()
print(f"Working directory: {os.getcwd()}")

Working directory: c:\Users\M Harshith\Desktop\NyayaAI\indialex-ft


In [2]:
!pip install -r requirements.txt --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\M Harshith\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [10]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads indialex-ft/.env

def _check_secret(name, required=True):
    val = os.environ.get(name)
    if val:
        print(f"{name} loaded: True")
    else:
        status = "MISSING (required — add to .env)" if required else "skipped (optional)"
        print(f"{name} loaded: {status}")

_check_secret("ANTHROPIC_API_KEY", required=True)
_check_secret("GROQ_API_KEY",      required=True)
_check_secret("HF_TOKEN",          required=True)   # needed: Llama 3.2 is a gated model
_check_secret("WANDB_API_KEY",     required=False)

ANTHROPIC_API_KEY loaded: True
GROQ_API_KEY loaded: True
HF_TOKEN loaded: True
WANDB_API_KEY loaded: skipped (optional)


In [ ]:
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("HuggingFace login successful.")
else:
    raise EnvironmentError(
        "HF_TOKEN is not set. Add it in the 🔑 Secrets panel on the left sidebar "
        "and enable 'Notebook access'."
    )

## Step 1: Baseline Evaluation

Runs inference on the test split using the **base model** (before fine-tuning).
Computes ROUGE-L and BERTScore F1, and saves results to `evals/baseline_results.json`.

In [12]:
!python scripts/baseline_eval.py

^C


## Step 2: Training

Fine-tunes the base model with **LoRA + QLoRA (4-bit NF4)** via TRL SFTTrainer.
Saves the LoRA adapter to `outputs/adapter/` and the merged model to `outputs/merged/`.
Logs metrics to Weights & Biases.

In [13]:
!python scripts/train.py

^C


## Step 3: Post-training Evaluation

Runs inference on the test split using the **fine-tuned merged model**.
Computes ROUGE-L and BERTScore F1, compares against baseline, and saves
a delta summary to `evals/ft_results.json`.

In [ ]:
!python scripts/evaluate.py

WARNING  Baseline file not found at c:\Users\M Harshith\Desktop\NyayaAI\indialex-ft\evals\baseline_results.json — skipping delta comparison.
INFO  Loading fine-tuned model from C:\Users\M Harshith\Desktop\NyayaAI\indialex-ft\outputs\indialex-ft\merged
Traceback (most recent call last):
  File "C:\Users\M Harshith\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\utils\hub.py", line 419, in cached_files
    hf_hub_download(
  File "C:\Users\M Harshith\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\utils\_validators.py", line 85, in _inner_fn
    validate_repo_id(arg_value)
  File "C:\Users\M Harshith\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\utils\_validators.py", line 135, in validate_repo_id
    raise HFValidati

## Step 4: LLM-as-Judge Evaluation

Uses **claude-sonnet-4-6** (Anthropic API) to score each fine-tuned answer on
four legal-domain dimensions (1–5 each): correctness, faithfulness, helpfulness,
and legal accuracy. Results saved to `evals/judge_results.json`.

In [ ]:
!python evals/llm_judge.py --results evals/ft_results.json

ERROR: evals\ft_results.json not found. Run evaluate.py (or baseline_eval.py) first.


In [ ]:
import shutil
from pathlib import Path

for name, base_dir in [("nyayaai_outputs", "outputs"), ("nyayaai_evals", "evals")]:
    if Path(base_dir).exists():
        shutil.make_archive(name, "zip", root_dir=".", base_dir=base_dir)
        print(f"Zipped {base_dir}/ → {name}.zip")
    else:
        print(f"Skipped {base_dir}/ — directory does not exist yet.")

try:
    from google.colab import files
    for f in ["nyayaai_outputs.zip", "nyayaai_evals.zip"]:
        if Path(f).exists():
            files.download(f)
except ImportError:
    print("Zip files saved locally (not running in Colab).")

Skipped outputs/ — directory does not exist yet.
Zipped evals/ → nyayaai_evals.zip
Zip files saved locally (not running in Colab).
